# YOLO11m training on 1080x1080 sliced dataset

This notebook clones the workspace, mounts Google Drive, extracts a YOLO-format dataset zip, slices 1920x1080 frames into 1080x1080 crops, prepares the local Ultralytics YOLO dataset layout, and starts YOLO11m training.

Expected dataset shape inside the zip:

```text
Dataset/
  Train/
    images/
    labels/
  Validation/
    images/
    labels/
```


In [ ]:
from pathlib import Path

# Required: set this to your workspace repo URL before running all cells.
REPO_URL = 'https://github.com/YOUR_USERNAME/obj-det-ws.git'
REPO_BRANCH = ''  # Optional. Leave empty for the default branch.

# Required: Google Drive zip path after drive.mount('/content/drive').
DATASET_ZIP_DRIVE_PATH = '/content/drive/MyDrive/Dataset.zip'

# Training outputs are written to Drive. Prepared sliced dataset stays local.
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/yolov11_runs'
RUN_NAME = 'yolo11m_datasetv1_sliced_1080_a100'

# Weights & Biases. Leave WANDB_API_KEY empty to use Colab Secret named WANDB_API_KEY or interactive login.
USE_WANDB = True
WANDB_PROJECT = 'obj-det-ws-yolov11'
WANDB_RUN_NAME = RUN_NAME
WANDB_ENTITY = ''
WANDB_API_KEY = ''

WORKSPACE_DIR = Path('/content/obj-det-ws')
LOCAL_DATASET_EXTRACT_DIR = Path('/content/dataset_raw')

CROP_SIZE = '1080x1080'
OVERLAP = 0.0

# A100 40GB defaults. Images are 1080x1080 crops, while YOLO training input defaults to 640.
EPOCHS = 100
IMGSZ = 640
BATCH = 64
DEVICE = '0'
WORKERS = 8
MODEL = 'yolo11m.pt'


In [ ]:
import os
import shutil
import subprocess
import sys

from google.colab import drive, userdata


def run(command, cwd=None):
    printable = ' '.join(str(part) for part in command)
    print(f'$ {printable}')
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)


if 'YOUR_USERNAME' in REPO_URL:
    raise ValueError('Set REPO_URL in the parameter cell before running the notebook.')

drive.mount('/content/drive')

if WORKSPACE_DIR.exists():
    shutil.rmtree(WORKSPACE_DIR)

clone_command = ['git', 'clone', '--depth', '1', '--recurse-submodules', '--shallow-submodules']
if REPO_BRANCH:
    clone_command.extend(['--branch', REPO_BRANCH])
clone_command.extend([REPO_URL, str(WORKSPACE_DIR)])
run(clone_command)

run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'wandb', 'PyYAML', 'Pillow'])
if USE_WANDB:
    run(['yolo', 'settings', 'wandb=True'])
else:
    run(['yolo', 'settings', 'wandb=False'])
run([sys.executable, '-m', 'py_compile', WORKSPACE_DIR / 'tools/slice_dataset.py', WORKSPACE_DIR / 'scripts/yolov11/train.py'])


In [ ]:
def get_colab_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


if USE_WANDB:
    import wandb

    api_key = WANDB_API_KEY or get_colab_secret('WANDB_API_KEY')
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()

    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_NAME'] = WANDB_RUN_NAME
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY

    print(f'W&B enabled: project={WANDB_PROJECT}, run={WANDB_RUN_NAME}')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled')


In [ ]:
import zipfile


def find_split(dataset_root, names):
    for name in names:
        candidate = dataset_root / name
        if (candidate / 'images').is_dir() and (candidate / 'labels').is_dir():
            return candidate
    return None


def find_yolo_dataset_root(search_root):
    candidates = [search_root]
    candidates.extend(path for path in search_root.rglob('*') if path.is_dir())
    for candidate in candidates:
        train_split = find_split(candidate, ['Train', 'train'])
        val_split = find_split(candidate, ['Validation', 'validation', 'Val', 'val', 'Valid', 'valid'])
        if train_split is not None and val_split is not None:
            return candidate, train_split, val_split
    raise FileNotFoundError('Could not find Dataset/Train/{images,labels} and Dataset/Validation/{images,labels}.')


dataset_source = Path(DATASET_ZIP_DRIVE_PATH)
if not dataset_source.exists():
    raise FileNotFoundError(f'Dataset path not found: {dataset_source}')

if LOCAL_DATASET_EXTRACT_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_EXTRACT_DIR)
LOCAL_DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if dataset_source.is_dir():
    dataset_search_root = dataset_source
else:
    if dataset_source.suffix.lower() != '.zip':
        raise ValueError('DATASET_ZIP_DRIVE_PATH must be a .zip file or an extracted dataset directory.')
    with zipfile.ZipFile(dataset_source) as archive:
        archive.extractall(LOCAL_DATASET_EXTRACT_DIR)
    dataset_search_root = LOCAL_DATASET_EXTRACT_DIR

dataset_root, train_source, val_source = find_yolo_dataset_root(dataset_search_root)
print(f'dataset_root: {dataset_root}')
print(f'train_source: {train_source}')
print(f'val_source: {val_source}')


In [ ]:
FINAL_YOLO_ROOT = WORKSPACE_DIR / 'datasets/datasetv1_sliced_1080/yolo'
TEMP_SLICED_ROOT = WORKSPACE_DIR / 'datasets/_sliced_yolo11m_tmp'

for path in [FINAL_YOLO_ROOT, TEMP_SLICED_ROOT]:
    if path.exists():
        shutil.rmtree(path)


def prepare_split(split_name, source_split):
    sliced_split_root = TEMP_SLICED_ROOT / split_name
    run([
        sys.executable,
        WORKSPACE_DIR / 'tools/slice_dataset.py',
        source_split,
        '-o', sliced_split_root,
        '--format', 'yolo',
        '--crop-size', CROP_SIZE,
        '--overlap', str(OVERLAP),
    ])

    image_target = FINAL_YOLO_ROOT / 'images' / split_name
    label_target = FINAL_YOLO_ROOT / 'labels' / split_name
    shutil.copytree(sliced_split_root / 'images', image_target, dirs_exist_ok=True)
    shutil.copytree(sliced_split_root / 'labels', label_target, dirs_exist_ok=True)
    return image_target, label_target


train_images, train_labels = prepare_split('train', train_source)
val_images, val_labels = prepare_split('val', val_source)
(FINAL_YOLO_ROOT / 'images' / 'test').mkdir(parents=True, exist_ok=True)
(FINAL_YOLO_ROOT / 'labels' / 'test').mkdir(parents=True, exist_ok=True)


def count_files(path, suffixes=None):
    if suffixes is None:
        return sum(1 for item in path.rglob('*') if item.is_file())
    return sum(1 for item in path.rglob('*') if item.is_file() and item.suffix.lower() in suffixes)


image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split_name, images_dir, labels_dir in [
    ('train', train_images, train_labels),
    ('val', val_images, val_labels),
]:
    image_count = count_files(images_dir, image_suffixes)
    label_count = count_files(labels_dir, {'.txt'})
    print(f'{split_name}: {image_count} images, {label_count} labels')
    if image_count == 0:
        raise ValueError(f'No sliced images found for {split_name}')
    if label_count != image_count:
        raise ValueError(f'Image/label count mismatch for {split_name}: {image_count} images, {label_count} labels')


In [ ]:
CONFIG_PATH = 'configs/yolov11/yolo11m_datasetv1_sliced_1080.yaml'
Path(OUTPUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)

TRAIN_COMMAND = [
    sys.executable,
    'scripts/yolov11/train.py',
    '--config', CONFIG_PATH,
    '--model', MODEL,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--device', DEVICE,
    '--workers', str(WORKERS),
    '--project', OUTPUT_DRIVE_DIR,
    '--name', RUN_NAME,
]

run([*TRAIN_COMMAND, '--dry-run'], cwd=WORKSPACE_DIR)


In [ ]:
run(TRAIN_COMMAND, cwd=WORKSPACE_DIR)
